# Neural Network Modeling
Chau Nguyen · MGMT 47500, Purdue University

Executable review workflow for the supervised autoencoder, temporal baselines, decile portfolios and factor interpretation. This notebook calls the tested modules; it contains no fabricated execution outputs. Historical saved outputs are in `notebooks/Historical_Decile_Analysis.ipynb`. See `RESULTS_PROVENANCE.md` for the distinction. Run from the repository root.

In [ ]:
from pathlib import Path
from data import load_panel
from analysis import run_backtest
from models import TrainConfig
import pandas as pd

DATA_DIR = Path("data/monthly")
OUTPUT_DIR = Path("artifacts")

## 1. Validate the data
The required sample is January 2001 through December 2021. Input checks enforce 94 characteristics, unique security-month keys, complete month coverage and finite observed measurements. Earlier-year archives cannot stand in for the stated experiment.

In [ ]:
panel = load_panel(DATA_DIR)
panel.groupby("year").size().to_frame("observations")

## 2. Fit and evaluate
For test year t, all model fitting uses data through t−2. Neural models select an epoch on t−1. Standardization is fit on training data only; contemporaneous imputation uses no target values. Defaults retain the documented architecture and loss weights. This is a full training run and may be expensive on CPU.

In [ ]:
summary = run_backtest(panel, OUTPUT_DIR, cfg=TrainConfig(), first_year=2011, last_year=2021, seed=42)
summary

## 3. Inspect temporal partitions and annual metrics

In [ ]:
display(pd.read_csv(OUTPUT_DIR / "split-audit.csv"))
display(pd.read_csv(OUTPUT_DIR / "metrics-by-year.csv"))

## 4. Evaluate top/bottom deciles
Long and short legs use the 90th and 10th prediction percentiles, respectively. Equal-weight returns include boundary ties; overlapping legs fail validation. Sharpe is monthly and returns are gross of costs.

In [ ]:
pd.read_csv(OUTPUT_DIR / "decile-portfolios.csv")

## 5. Interpret the encoder
The product of the two encoder weight matrices is an approximation because of ReLU nonlinearity. These loadings describe the last evaluated encoder. Factor-premium regressions are descriptive, not tradable forecasts.

In [ ]:
display(pd.read_csv(OUTPUT_DIR / "factor-loadings.csv"))
display(pd.read_csv(OUTPUT_DIR / "factor-premia-2021.csv"))

## 6. Review reproducibility metadata
`run-info.json` records the sample, configuration, seed, versions and metric definitions. The command-line entry point additionally hashes each monthly input file.

In [ ]:
import json
json.loads((OUTPUT_DIR / "run-info.json").read_text())